# Transformación de datos en tiempo real para PyPOTS

In [1]:
import duckdb
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# DuckDB raw data
root_dir = os.environ.get("PROJECT_ROOT", "")
db_file = os.environ.get("DDBB_PATH", "duck_db/occupancy.duckdb")
db_path = os.path.join(root_dir, db_file)

# Export processed parquet
data_folder = os.environ.get("DATA_FOLDER_PATH_IPYNB", "../../data/")
df_model_name = os.environ.get("PARQUET_API_MODEL", "df_pypots_api.parquet")
output_path = os.path.join(data_folder, df_model_name)

## 1. Carga de datos desde DuckDB

In [3]:
conn = duckdb.connect(db_path, read_only=True)
query = """
    SELECT 
        kerbsideid AS DeviceId,
        CAST(status_timestamp AS TIMESTAMP) AS Timestamp,
        status_description,
        location.lon AS lon,
        location.lat AS lat
    FROM occupancy
    WHERE status_timestamp IS NOT NULL
"""
df = conn.execute(query).fetchdf()
conn.close()

In [4]:
print("Muestra del dataset inicial crudo:")
print(df.head())
print(f"\nDimensiones iniciales del dataset: {df.shape}")
print(df.isnull().sum())

Muestra del dataset inicial crudo:
   DeviceId           Timestamp status_description         lon        lat
0      8888 2024-10-10 21:40:32            Present  144.959930 -37.804936
1      8891 2024-10-10 23:53:04            Present  144.959859 -37.804783
2      9623 2026-05-01 12:35:45         Unoccupied  144.948613 -37.804972
3      9628 2026-05-01 12:27:02         Unoccupied  144.948459 -37.804955
4      9636 2024-06-27 07:52:31         Unoccupied  144.948324 -37.805044

Dimensiones iniciales del dataset: (11078103, 5)
DeviceId              0
Timestamp             0
status_description    0
lon                   0
lat                   0
dtype: int64


## 2. Análisis estadístico

In [5]:

START_DATE = pd.Timestamp(os.environ.get("START_DATE", "2026-05-01 00:00:00"))
END_DATE = pd.Timestamp(os.environ.get("END_DATE", "2026-05-31 23:59:59"))

mask_active_month = (
    (df['Timestamp'] >= START_DATE)
    &
    (df['Timestamp'] <= END_DATE)
)

active_devices = df.loc[
    mask_active_month,
    'DeviceId'
].unique()

print(f"\nSensores activos en mayo: {len(active_devices)}")


Sensores activos en mayo: 1601


In [6]:
# check distribution of interval gaps for active devices
df_active = df[
    df['DeviceId'].isin(active_devices)
].copy()


df_events = (
    df_active[['DeviceId', 'Timestamp']]
    .drop_duplicates()
    .sort_values(['DeviceId', 'Timestamp'])
)

df_events['Gap'] = (
    df_events.groupby('DeviceId')['Timestamp']
    .diff()
)

gaps = df_events['Gap'].dropna()

print("\nDistribución global de gaps:")
print(
    gaps.describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)


Distribución global de gaps:
count                   1022416
mean     0 days 01:18:15.937576
std      0 days 05:04:12.542756
min             0 days 00:00:01
50%             0 days 00:30:08
90%             0 days 02:40:14
95%             0 days 04:44:53
99%      0 days 11:31:05.849999
99.5%    0 days 15:01:37.775000
99.9%    1 days 10:45:33.484999
max            76 days 04:56:41
Name: Gap, dtype: object


In [7]:
# See which sensors have the largest gaps
max_gap_sensor = (
    df_events.groupby('DeviceId')['Gap']
    .max()
    .dropna()
)

print("\nTop sensores con mayores gaps:")
print(
    max_gap_sensor
    .sort_values(ascending=False)
    .head(20)
)


Top sensores con mayores gaps:
DeviceId
61520   76 days 04:56:41
23386   65 days 20:41:17
13469   53 days 04:26:33
13474   41 days 19:39:51
13466   40 days 16:08:43
10904   38 days 16:11:45
54518   32 days 22:34:41
66459   31 days 09:45:12
13475   30 days 22:30:56
60245   28 days 11:34:48
18532   27 days 08:54:18
18534   25 days 05:58:20
62657   23 days 20:12:34
11929   22 days 22:59:03
53228   20 days 21:38:26
17330   19 days 00:11:31
13467   15 days 19:49:37
56710   15 days 03:49:29
56716   15 days 02:31:09
56717   15 days 01:52:55
Name: Gap, dtype: timedelta64[us]


## 3. Transformación de datos

In [8]:
df['VehiclePresent'] = df['status_description'].apply(lambda x: 1.0 if x == 'Present' else 0.0) # map target to binary values
df.drop(columns=['status_description'], inplace=True)

# Avoid old sensors returned by the API, only get the data from the established date range
df_time = df[df['Timestamp'] <= END_DATE].copy() # only filter by end_date to keep data prior to the start date for forward fill

# Remove duplicates and sort
df_time = df_time.drop_duplicates(subset=['DeviceId', 'Timestamp'], keep='last')
df_time = df_time.sort_values(by=['DeviceId', 'Timestamp'])

print(df_time.shape)

(890549, 5)


In [9]:
# Build 15 min grid
grid_freq = os.environ.get("GRID_FREQ", "15min")
time_grid = pd.date_range(start=START_DATE, end=END_DATE, freq=grid_freq) # filter by the established date range
unique_devices = df_time['DeviceId'].unique()
full_index = pd.MultiIndex.from_product([unique_devices, time_grid], names=['DeviceId', 'Timestamp'])

# Apply forward fill in next time freq if there is no state update
df_indexed = df_time.set_index(['DeviceId', 'Timestamp'])
df_continuous = df_indexed.reindex(full_index, method='ffill').reset_index()
print(df_continuous.shape)

(9933888, 5)


In [10]:
# Apply same trigonometric transformations as in the historic dataset
df_continuous['Hour'] = df_continuous['Timestamp'].dt.hour + df_continuous['Timestamp'].dt.minute / 60.0
df_continuous['Hour_sin'] = np.sin(2 * np.pi * df_continuous['Hour'] / 24.0)
df_continuous['Hour_cos'] = np.cos(2 * np.pi * df_continuous['Hour'] / 24.0)

df_continuous['WeekDay'] = df_continuous['Timestamp'].dt.dayofweek
df_continuous['WeekDay_sin'] = np.sin(2 * np.pi * df_continuous['WeekDay'] / 7.0)
df_continuous['WeekDay_cos'] = np.cos(2 * np.pi * df_continuous['WeekDay'] / 7.0)

df_continuous['Month'] = df_continuous['Timestamp'].dt.month
df_continuous['Month_sin'] = np.sin(2 * np.pi * df_continuous['Month'] / 12.0)
df_continuous['Month_cos'] = np.cos(2 * np.pi * df_continuous['Month'] / 12.0)

df_continuous.drop(columns=['Hour', 'WeekDay', 'Month'], inplace=True)
print(df_continuous.shape)

(9933888, 11)


In [11]:
# Reorder columns to match the historic dataframe structure 
column_order = ['DeviceId', 'Timestamp', 'VehiclePresent', 'lon', 'lat', 
                'Hour_sin', 'Hour_cos', 'WeekDay_sin', 'WeekDay_cos', 'Month_sin', 'Month_cos']
df_final = df_continuous[column_order]

In [12]:
print("\nMuestra del dataset final procesado:")
print(f"\nDimensiones finales del dataset: {df_final.shape}")
print(df_final.head())
print(df_final.tail())


Muestra del dataset final procesado:

Dimensiones finales del dataset: (9933888, 11)
   DeviceId           Timestamp  VehiclePresent        lon        lat  \
0      5691 2026-05-01 00:00:00             0.0  144.95633 -37.814096   
1      5691 2026-05-01 00:15:00             0.0  144.95633 -37.814096   
2      5691 2026-05-01 00:30:00             0.0  144.95633 -37.814096   
3      5691 2026-05-01 00:45:00             0.0  144.95633 -37.814096   
4      5691 2026-05-01 01:00:00             0.0  144.95633 -37.814096   

   Hour_sin  Hour_cos  WeekDay_sin  WeekDay_cos  Month_sin  Month_cos  
0  0.000000  1.000000    -0.433884    -0.900969        0.5  -0.866025  
1  0.065403  0.997859    -0.433884    -0.900969        0.5  -0.866025  
2  0.130526  0.991445    -0.433884    -0.900969        0.5  -0.866025  
3  0.195090  0.980785    -0.433884    -0.900969        0.5  -0.866025  
4  0.258819  0.965926    -0.433884    -0.900969        0.5  -0.866025  
         DeviceId           Timestamp  Vehi

## 4. Exportar a parquet

In [13]:
df_final.to_parquet(output_path, index=False)
print(f"Dataframe exportado a {output_path}")

Dataframe exportado a ../../data/df_pypots_api.parquet
